In [40]:
import pandas as pd
import numpy as np
import yaml
import os

from MAST_tools.utils.path_utils import PACKAGE_METADATA_DIR
from preproc_paths import ( 
    DEFAULT_SHOTS_STATS_TRAIN_FILE, 
    DEFAULT_SHOTS_STATS_VAL_FILE, 
    DEFAULT_SHOTS_STATS_TEST_FILE,
    DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE
)

## Get global mean and std for stdscaling

In [41]:
df_train = pd.read_csv(DEFAULT_SHOTS_STATS_TRAIN_FILE).sort_values(by=["shot_idx", "variable"])

In [42]:
df_train[df_train["shot_id"]==23812].head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median
305713,7838,23812,equilibrium-beta_normal,129.0,39.0,1.736305,0.680695,-0.584141,3.277398,1.600326
305712,7838,23812,equilibrium-beta_pol,129.0,39.0,0.455106,0.046316,-0.092767,0.877834,0.400647
305711,7838,23812,equilibrium-beta_tor,129.0,39.0,3.164488,2.610983,-1.258600,6.050639,2.934588
305715,7838,23812,equilibrium-bphi_rmag,129.0,39.0,-0.535350,0.005481,-0.969970,-0.477782,-0.507632
305714,7838,23812,equilibrium-bvac_rmag,129.0,39.0,-0.486946,0.002470,-0.609947,-0.444663,-0.460203


In [43]:
# GLOBAL MEAN BASED ON TRAIN
global_mean = (
    df_train
    .groupby("variable")[["n_dim_shot", "mean", ]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
df_train_with_group_mean = df_train.join(global_mean, on="variable")

# GLOBAL VARIANCE BASED ON TRAIN
global_variance = (
    df_train_with_group_mean
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)
df_train_with_group_mean_and_variance = df_train_with_group_mean.join(global_variance, on="variable")
df_train_with_group_mean_and_variance.head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance
31,0,21719,equilibrium-beta_normal,109.0,36.0,0.558459,0.064401,0.080094,0.957282,0.693243,1.109633,1.563052
30,0,21719,equilibrium-beta_pol,109.0,36.0,0.137247,0.003788,0.026632,0.339347,0.161548,0.231777,0.079105
29,0,21719,equilibrium-beta_tor,109.0,36.0,1.125700,0.310402,0.087273,1.921715,1.152199,3.178590,196.100433
33,0,21719,equilibrium-bphi_rmag,109.0,36.0,-0.551871,0.000863,-0.635485,-0.498496,-0.548581,-0.536505,0.004684
32,0,21719,equilibrium-bvac_rmag,109.0,36.0,-0.471889,0.001365,-0.620015,-0.445769,-0.457897,-0.465714,0.003623


#### Remove z-6 outliers from computation

In [44]:
# Z-SCORE COMPUTATION TRAIN
# df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) / np.sqrt(df_train_with_group_mean_and_variance["n_dim_shot"]) )
df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) )

df_train_with_group_mean_and_variance

# COUNTING OUTLIERS
print( f'Count of z-6 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 6 )} out of {len(df_train_with_group_mean_and_variance)}')
print( f'Count of z-12 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 12 )} out of {len(df_train_with_group_mean_and_variance)}')

df_train_with_group_mean_and_variance["outlier_z_6"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 6 
df_train_with_group_mean_and_variance["outlier_z_12"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 12

df_train_extended = df_train_with_group_mean_and_variance.copy()
df_train_with_group_mean_and_variance.sort_values('z_score').dropna()

Count of z-6 outliers 46 out of 349557
Count of z-12 outliers 31 out of 349557


,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
97416,2497,15926,equilibrium-bphi_rmag,54.0,35.0,-1.937142e+00,5.555050e-01,-2.822751e+00,-5.295477e-01,-2.357045e+00,-5.365051e-01,4.684335e-03,-20.464524,True,True
97413,2497,15926,equilibrium-beta_pol,54.0,35.0,-3.955077e+00,5.966282e+00,-7.431953e+00,5.293262e-03,-5.287639e+00,2.317772e-01,7.910513e-02,-14.886256,True,True
97414,2497,15926,equilibrium-beta_normal,54.0,35.0,-1.491562e+01,8.302890e+01,-2.785353e+01,2.150156e-02,-1.984107e+01,1.109633e+00,1.563052e+00,-12.817934,True,True
246981,6332,17404,equilibrium-bphi_rmag,62.0,45.0,-1.333206e+00,6.021557e-01,-2.879354e+00,-5.473692e-01,-1.202947e+00,-5.365051e-01,4.684335e-03,-11.640496,True,False
246978,6332,17404,equilibrium-beta_pol,62.0,45.0,-3.030435e+00,1.729355e+01,-1.168474e+01,1.076197e-01,-2.117036e+00,2.317772e-01,7.910513e-02,-11.598714,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57476,1473,12224,equilibrium-beta_tor,121.0,36.0,2.781161e+02,3.664073e+04,1.492902e+01,5.728040e+02,2.067096e+02,3.178590e+00,1.961004e+02,19.633364,True,True
226385,5804,12223,equilibrium-beta_tor,122.0,33.0,2.865881e+02,3.673864e+04,-1.621598e+01,5.552874e+02,2.507517e+02,3.178590e+00,1.961004e+02,20.238352,True,True
153611,3938,12232,equilibrium-beta_tor,124.0,38.0,3.167945e+02,4.333140e+04,1.421999e+01,6.321520e+02,2.939886e+02,3.178590e+00,1.961004e+02,22.395396,True,True
189305,4853,20038,thomson_scattering-n_e,7200.0,5003.0,1.373060e+21,5.138213e+42,1.192202e+17,1.136142e+22,3.424352e+19,2.262173e+19,3.423544e+39,23.080046,True,True


In [45]:
# import matplotlib.pyplot as plt

# plt.hist(df_train_with_group_mean_and_variance["z_score"], bins=30)
# plt.xlabel("z_score")
# plt.ylabel("Frequency")
# plt.title("Histogram of z-scores")
# plt.show()

In [46]:
df_train_extended.head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
31,0,21719,equilibrium-beta_normal,109.0,36.0,0.558459,0.064401,0.080094,0.957282,0.693243,1.109633,1.563052,-0.440862,False,False
30,0,21719,equilibrium-beta_pol,109.0,36.0,0.137247,0.003788,0.026632,0.339347,0.161548,0.231777,0.079105,-0.336099,False,False
29,0,21719,equilibrium-beta_tor,109.0,36.0,1.125700,0.310402,0.087273,1.921715,1.152199,3.178590,196.100433,-0.146597,False,False
33,0,21719,equilibrium-bphi_rmag,109.0,36.0,-0.551871,0.000863,-0.635485,-0.498496,-0.548581,-0.536505,0.004684,-0.224510,False,False
32,0,21719,equilibrium-bvac_rmag,109.0,36.0,-0.471889,0.001365,-0.620015,-0.445769,-0.457897,-0.465714,0.003623,-0.102591,False,False


In [47]:
# OG global mean and variance
global_mean = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
global_variance = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)

# MEAN AND VARIANCE COMPUTATION WITHOUT OUTLIERS Z-6
# global mean and variance without the 6-z outliers
global_mean_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean_no_z_6")
)

df_train_extended = df_train_extended.join(global_mean_no_z_6, on="variable")
global_variance_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean_no_z_6"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean_no_z_6"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance_no_z_6")
)


In [48]:
# Merge all the stats into a single DataFrame
df_stats = pd.DataFrame({
    "variable": global_mean.index,
    # "mean_all": global_mean.values,
    # "std_all": np.sqrt(global_variance.values),
    "mean_no_outliers_z6": global_mean_no_z_6.values,
    "std_no_outliers_z6": np.sqrt(global_variance_no_z_6.values),
})

# Build the dictionary for YAML
final_dict = {}
for _, row in df_stats.iterrows():
    var = row["variable"]
    final_dict[var] = {
        "mean": {
            # "all": row["mean_all"],
            "no_outliers_z6": row["mean_no_outliers_z6"],
            # "no_outliers_z12": row["mean_no_outliers_z12"]
        },
        "std": {
            # "all": row["std_all"],
            "no_outliers_z6": row["std_no_outliers_z6"],
            # "no_outliers_z12": row["std_no_outliers_z12"]
        }
    }

# Write to YAML
with open(DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE, "w") as f:
    yaml.dump(final_dict, f, sort_keys=False)


## Remove outliers from train, val, test

In [49]:
df_train = pd.read_csv(DEFAULT_SHOTS_STATS_TRAIN_FILE).sort_values(by=["shot_idx", "variable"])
df_val = pd.read_csv(DEFAULT_SHOTS_STATS_VAL_FILE).sort_values(by=["shot_idx", "variable"])
df_test = pd.read_csv(DEFAULT_SHOTS_STATS_TEST_FILE).sort_values(by=["shot_idx", "variable"])

df_all = pd.concat([df_train, df_val, df_test])

In [50]:
df_all_with_stats = df_all.join(global_mean_no_z_6, on="variable").join(global_variance_no_z_6, on="variable")
df_all_with_stats

df_all_with_stats["z_score"] = (df_all_with_stats["mean"] - df_all_with_stats["global_mean_no_z_6"]) / ( np.sqrt(df_all_with_stats["global_variance_no_z_6"]) )

df_all_with_stats["outlier_z_12"] = abs(df_all_with_stats["z_score"]) > 12

print( f'Count of z-12 outliers {sum ( df_all_with_stats["outlier_z_12"] ) } out of {len(df_all_with_stats)}')
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() )} variables {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() } (out of {len( df_all_with_stats["variable"].unique()) })' )
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique() )} shots {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique()} (out of {len( df_all_with_stats["shot_id"].unique()) })' )


Count of z-12 outliers 45 out of 436332
Spanning 8 variables ['equilibrium-beta_tor' 'equilibrium-x_point_r' 'pf_active-coil_voltage'
 'soft_x_rays-horizontal_cam_upper' 'thomson_scattering-n_e'
 'equilibrium-beta_normal' 'equilibrium-beta_pol' 'equilibrium-bphi_rmag'] (out of 39)
Spanning 43 shots [12214 15741 25456 13298 25454 12224 12210 20111 12221 12196 25455 15926
 19461 20166 25453 12232 12216 12219 12209 19953 12230 12218 20038 12213
 20271 19454 12223 19453 12231 17404 12220 12215 12237 12208 12204 19460
 20152 20124 19450 12229 25457 12203 19455] (out of 11188)


In [51]:
# import matplotlib.pyplot as plt

# plt.hist(
#     df_all_with_stats[df_all_with_stats["outlier_z_12"]]["shot_id"],
#     bins=100,
#     color="salmon",
#     edgecolor="black"
# )
# plt.xlabel("shot_id")
# plt.ylabel("Number of outliers")
# plt.show()


In [58]:
zscore_outlier = (
    df_all_with_stats[df_all_with_stats["outlier_z_12"]].groupby("shot_id")["variable"]
    .apply(list)
    .to_dict()
)
print(zscore_outlier)

with open("dict_zscore_outlier.yaml", "w") as f_:
    yaml.dump(zscore_outlier, f_, sort_keys=False)

{12196: ['equilibrium-beta_tor'], 12203: ['equilibrium-beta_tor'], 12204: ['equilibrium-beta_tor'], 12208: ['equilibrium-beta_tor'], 12209: ['equilibrium-beta_tor'], 12210: ['equilibrium-beta_tor'], 12213: ['equilibrium-beta_tor'], 12214: ['equilibrium-beta_tor'], 12215: ['equilibrium-beta_tor'], 12216: ['equilibrium-beta_tor'], 12218: ['equilibrium-beta_tor'], 12219: ['equilibrium-beta_tor'], 12220: ['equilibrium-beta_tor'], 12221: ['equilibrium-beta_tor'], 12223: ['equilibrium-beta_tor'], 12224: ['equilibrium-beta_tor'], 12229: ['equilibrium-beta_tor'], 12230: ['equilibrium-beta_tor'], 12231: ['equilibrium-beta_tor'], 12232: ['equilibrium-beta_tor'], 12237: ['equilibrium-beta_tor'], 13298: ['soft_x_rays-horizontal_cam_upper'], 15741: ['equilibrium-x_point_r'], 15926: ['equilibrium-beta_normal', 'equilibrium-beta_pol', 'equilibrium-bphi_rmag'], 17404: ['equilibrium-bphi_rmag'], 19450: ['soft_x_rays-horizontal_cam_upper'], 19453: ['soft_x_rays-horizontal_cam_upper'], 19454: ['soft_x_ra

## Combine manual and zscore outliers

In [60]:
import yaml

with open(os.path.join(PACKAGE_METADATA_DIR, "dict_manual_outlier.yaml"), "r") as f:
    manual_outlier = yaml.safe_load(f)

print(manual_outlier)

{11827: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11830: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11939: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11940: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11941: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11942: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11943: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11946: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11996: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11998: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12387: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12388: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12745: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13041: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13042: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13043: ['soft_x_rays-hor

In [61]:
manual_outlier

{11827: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11830: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11939: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11940: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11941: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11942: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11943: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11946: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11996: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11998: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12387: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12388: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12745: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 13041: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 13042: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 130

In [62]:
from collections import defaultdict

combined = defaultdict(set)

for d in (zscore_outlier, manual_outlier):
    for k, v in d.items():
        combined[k].update(v)

combined_outlier = {k: list(v) for k, v in combined.items()}

In [63]:
with open("dict_outlier_metadata.yaml", "w") as f_:
    yaml.dump(combined_outlier, f_, sort_keys=False)